# ML-09 · Validation and Research Claim Audit
## 1. Two findings and methodology questions
[FlyRank research report](https://state-of-seo-2026.flyrank.ai/)
- **Finding 3, The CTR Cliff.** How sensitive are the position-bucket comparisons to impression weighting, client mix and minimum exposure? I would compare pooled CTR to per-client summaries and inspect uncertainty for low-volume buckets.
- **Finding 4, The Freshness Multiplier.** Were refreshed and unrefreshed pages similar before intervention? I would check how refresh timing was defined, compare prior trajectories and account for seasonality before interpreting an association as an effect.

## 2. Before and after
Re-run the Week 5 logistic model on the same starter snapshot, same queue eligibility and same label: a loss of more than 20% of impressions in the next 30 days. A row-random split is the diagnostic comparison. The after design repeats the Week 5 client holdout to block client overlap. The test populations differ, so base rates and baseline-relative gains matter alongside absolute metrics. Neither split is a later-date evaluation.

In [1]:
from pathlib import Path
import json
import pandas as pd
from IPython.display import display

relative_input = Path("data/raw/content_refresh_anonymized.csv")
here = Path.cwd()
root = next((p for p in (here, *here.parents) if (p / relative_input).is_file()), None)
if root is None:
    from importlib.util import find_spec
    from hashlib import sha256
    from urllib.request import urlopen

    if find_spec("google") is None or find_spec("google.colab") is None:
        raise FileNotFoundError(f"Run inside the starter repository containing {relative_input}")

    root = here
    data_url = (
        "https://raw.githubusercontent.com/mhv-codes/flyrank-ml-internship/"
        "48d0dac47169e9f14b106037938d414b16529aaf/"
        f"{relative_input}"
    )
    with urlopen(data_url, timeout=60) as response:
        starter_csv = response.read()
    expected_sha256 = "c43bdac4eccfa17fcd8a33974fe36f2c998c03a3ae3af8d80cf712abab5d6396"
    if sha256(starter_csv).hexdigest() != expected_sha256:
        raise ValueError("Downloaded starter CSV does not match the verified repository copy.")
    input_path = root / relative_input
    input_path.parent.mkdir(parents=True, exist_ok=True)
    input_path.write_bytes(starter_csv)

import numpy as np
import sklearn
from sklearn.model_selection import GroupShuffleSplit
from sklearn.preprocessing import StandardScaler, FunctionTransformer
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, roc_auc_score

keys = ["client_id", "content_id"]
raw_columns = keys + ["content_age_days", "impressions_prev_30d", "clicks_prev_30d", "impressions_last_30d"]
data = pd.read_csv(root / relative_input, usecols=raw_columns)
assert len(data) == 30000 and not data.duplicated(keys).any()
data["age_at_cutoff"] = data["content_age_days"] - 30
pool = data.loc[data.age_at_cutoff.ge(30) & data.impressions_prev_30d.ge(500)].copy()
features = ["impressions_prev_30d", "clicks_prev_30d", "age_at_cutoff"]
assert pool[keys + features + ["impressions_last_30d"]].notna().all().all()
assert pool[features].ge(0).all().all()
pool["decline"] = (5 * pool.impressions_last_30d < 4 * pool.impressions_prev_30d).astype(int)
print(f"pandas {pd.__version__}, scikit-learn {sklearn.__version__}; queue candidates: {len(pool):,}")


pandas 2.2.3, scikit-learn 1.8.0; queue candidates: 11,119


In [2]:
from sklearn.model_selection import train_test_split
from sklearn.base import clone
from sklearn.inspection import permutation_importance

seed = 42
row_train, row_test = train_test_split(pool, test_size=.25, random_state=seed, stratify=pool.decline)
gi, gj = next(GroupShuffleSplit(n_splits=1, test_size=.25, random_state=seed).split(pool, groups=pool.client_id))
group_train, group_test = pool.iloc[gi], pool.iloc[gj]
estimator = make_pipeline(FunctionTransformer(np.log1p), StandardScaler(),
                          LogisticRegression(max_iter=1000, random_state=seed))

def evaluate(design, train, test, columns):
    model = clone(estimator).fit(train[columns], train.decline)
    scored = test.assign(model_score=model.predict_proba(test[columns])[:,1],
                         baseline_score=test.impressions_prev_30d)
    overlap = len(set(train.client_id) & set(test.client_id))
    rows = []
    for method, col in [("Week 4 rule","baseline_score"),("Logistic regression","model_score")]:
        ranked = scored.sort_values([col,*keys],ascending=[False,True,True])
        rows.append(dict(split=design, method=method, rows=len(test), clients=test.client_id.nunique(),
                         overlap=overlap, base_rate=test.decline.mean(), P10=ranked.head(10).decline.mean(),
                         P50=ranked.head(50).decline.mean(), AP=average_precision_score(test.decline,scored[col]),
                         AUC=roc_auc_score(test.decline,scored[col])))
    return rows, model, scored.sort_values(["model_score",*keys],ascending=[False,True,True])

r1,_,_ = evaluate("Row random",row_train,row_test,features)
r2,model,ranked = evaluate("Client holdout",group_train,group_test,features)
assert r1[0]["overlap"] > 0 and r2[0]["overlap"] == 0
print(pd.DataFrame(r1+r2).round(4).to_string(index=False))
for rows in (r1,r2):
    print(rows[0]["split"],"model minus rule P@50:",round(rows[1]["P50"]-rows[0]["P50"],3))


         split              method  rows  clients  overlap  base_rate  P10  P50     AP    AUC
    Row random         Week 4 rule  2780       25       25     0.6083  0.2 0.40 0.5663 0.4506
    Row random Logistic regression  2780       25       25     0.6083  0.7 0.76 0.7505 0.6796
Client holdout         Week 4 rule   783        7        0     0.9106  0.8 0.92 0.9309 0.6205
Client holdout Logistic regression   783        7        0     0.9106  1.0 0.98 0.9483 0.6463
Row random model minus rule P@50: 0.36
Client holdout model minus rule P@50: 0.06


## 3. Leakage audit and failures
Timeline: days 31–60 back provide prior clicks and impressions. At the cutoff, creation age is rolled back 30 days. The subsequent 30 days supply the outcome only. Queue eligibility uses prior impressions and age. The published snapshot itself may have selection effects we cannot audit from these columns. Exclude 90-day totals and rates that overlap the outcome, latest-window metrics, label-derived trend fields, product decisions, and IDs. IDs are used solely for grouping and ties. The following deliberately invalid control checks whether adding the answer can inflate performance. It is excluded from every honest result.

In [3]:
assert features == ["impressions_prev_30d","clicks_prev_30d","age_at_cutoff"]
bad_train = group_train.assign(leak=group_train.decline)
bad_test = group_test.assign(leak=group_test.decline)
bad = clone(estimator).fit(bad_train[features+["leak"]],bad_train.decline)
print("INVALID label-copy control AUC:", round(roc_auc_score(bad_test.decline,
      bad.predict_proba(bad_test[features+["leak"]])[:,1]),4))
imp = permutation_importance(model,group_test[features],group_test.decline,
                             scoring="average_precision",n_repeats=8,random_state=seed)
print("Held-out AP drop when shuffled:")
print(pd.Series(imp.importances_mean,index=features).round(4).to_string())
ranked = ranked.assign(review_rank=np.arange(1,len(ranked)+1))
cases=pd.concat([ranked.head(50).query("decline == 0").head(2),
                 ranked.iloc[50:].query("decline == 1").head(2)]).head(3)
assert len(cases)==3
for i,r in enumerate(cases.itertuples(),1):
    change=100*(r.impressions_last_30d/r.impressions_prev_30d-1)
    print(f"Case {i}: rank {r.review_rank}, prior exposure {r.impressions_prev_30d}, prior clicks {r.clicks_prev_30d}, observed change {change:+.1f}%")
print("False picks use review slots without meeting the decline cutoff. Missed cases outside the budget can still have steep declines. Check seasonality, demand and page history before recommending edits.")


INVALID label-copy control AUC: 1.0
Held-out AP drop when shuffled:
impressions_prev_30d    0.0139
clicks_prev_30d         0.0028
age_at_cutoff           0.0301
Case 1: rank 43, prior exposure 528, prior clicks 0, observed change -15.5%
Case 2: rank 51, prior exposure 1014, prior clicks 0, observed change -75.0%
Case 3: rank 52, prior exposure 4171, prior clicks 1, observed change -78.9%
False picks use review slots without meeting the decline cutoff. Missed cases outside the budget can still have steep declines. Check seasonality, demand and page history before recommending edits.


## 4. Claim rewrite
Replace “the model finds pages that should be refreshed” with “on seven held-out clients in this snapshot, the model ranked 49 observed declines in its first 50 review slots, compared with 46 for the exposure rule.” The holdout has a higher decline rate than training and represents relatively few pages. The difference is directional. Permutation importance measures sensitivity to shuffling in this test set. It cannot establish causes, and correlated clicks and impressions complicate individual attributions. Review priority is decision support. Test an untouched later warehouse window and compare multiple client folds before deployment.

## 5. Self-check
Both methods share the same test rows within each design. The scaler and model are fit on training rows only. The grouped test has zero overlap. No gated data, client identifiers or row-level URLs are published.